In [1]:
# Self-contained notebook cell: monthly cumulative animated scatter for hard boulder ascents
import re
from io import StringIO
from datetime import datetime

import pandas as pd
import requests
from dateutil import parser as dateparser

import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

URL = "https://climbing-history.org/list/14/hard-boulder-problem-ascents"

# ---------- Parsing helpers ----------
def clean_date(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"(\d+)(st|nd|rd|th)\b", r"\1", s, flags=re.IGNORECASE)
    return s

def parse_date(s: str):
    if pd.isna(s) or str(s).strip() == "":
        return pd.NaT

    s = clean_date(s)

    m = re.match(r"(?i)between\s+(.+?)\s+and\s+(.+)$", s)
    if m:
        return dateparser.parse(m.group(1).strip(), fuzzy=True, dayfirst=True)

    m = re.match(r"(?i)before\s+(.+)$", s)
    if m:
        return dateparser.parse(m.group(1).strip(), fuzzy=True, dayfirst=True)

    if re.fullmatch(r"\d{4}", s):
        return datetime(int(s), 1, 1)

    if re.fullmatch(r"[A-Za-z]{3,9}\s+\d{4}", s):
        dt = dateparser.parse(s, fuzzy=True)
        return datetime(dt.year, dt.month, 1)

    return dateparser.parse(s, fuzzy=True, dayfirst=True)

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

def month_label(m):
    return pd.Timestamp(m).strftime("%Y-%m")

def normalize_colname(c):
    c = str(c).strip()
    c = re.sub(r"\s+", " ", c)
    return c

# ---------- Grade setup ----------
grade_to_y = {
    "8C+": 17,
    "9A": 18,
    "9A+": 19,
}
grade_order = ["8C+", "9A", "9A+"]

grade_colors = {
    "8C+": "#4cc9f0",
    "9A":  "#c300ff",
    "9A+": "#f72585",
}

base_sizes = {
    "8C+": 7,
    "9A": 10,
    "9A+": 13,
}

# ---------- Load table ----------
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
tables = pd.read_html(StringIO(html), flavor="bs4")

if not tables:
    raise ValueError("No tables found on the page.")

df = tables[0].copy()

# Clean column names
df.columns = [normalize_colname(c) for c in df.columns]

# Helpful debug
print("Raw columns:", list(df.columns))

# Robust column mapping
col_map = {}
for c in df.columns:
    lc = c.lower()
    if "climber" in lc:
        col_map[c] = "Climber"
    elif "climb" in lc:
        col_map[c] = "Climb"
    elif "grade" in lc:
        col_map[c] = "Grade"
    elif "date" in lc:
        col_map[c] = "Date"

df = df.rename(columns=col_map)

required = ["Climber", "Climb", "Grade", "Date"]
missing = [c for c in required if c not in df.columns]

# Fallback: if pandas didn't parse headers right, force first 4 columns
if missing and df.shape[1] >= 4:
    df = df.iloc[:, :4].copy()
    df.columns = ["Climber", "Climb", "Grade", "Date"]
    missing = []

if missing:
    raise ValueError(f"Could not identify expected columns. Got: {list(df.columns)}")

df = df[["Climber", "Climb", "Grade", "Date"]].copy()

df["date_parsed"] = df["Date"].apply(parse_date)
df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["Y"] = df["Grade_clean"].map(grade_to_y)

df = df.dropna(subset=["date_parsed", "Y"]).sort_values("date_parsed").reset_index(drop=True)

df["month"] = df["date_parsed"].dt.to_period("M").dt.to_timestamp()
months = sorted(df["month"].unique())

df["cumulative_rank"] = range(1, len(df) + 1)

# ---------- Trace builder ----------
def make_trace(d: pd.DataFrame, grade: str):
    dd = d[d["Grade_clean"] == grade]
    return go.Scatter(
        x=dd["date_parsed"],
        y=dd["Y"],
        mode="markers",
        name=grade,
        marker=dict(
            size=base_sizes.get(grade, 8),
            opacity=0.90,
            color=grade_colors.get(grade, "#FFFFFF"),
        ),
        customdata=list(
            zip(
                dd["Climber"],
                dd["Climb"],
                dd["Grade_clean"],
                dd["Date"],
                dd["cumulative_rank"],
            )
        ),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "%{customdata[1]}<br>"
            "Grade: %{customdata[2]}<br>"
            "Date: %{customdata[3]}<br>"
            "Cumulative ascent #: %{customdata[4]}"
            "<extra></extra>"
        ),
        showlegend=True,
    )

# ---------- Initial state ----------
initial_month = months[-1]
d0 = df[df["month"] <= initial_month]

initial_traces = [make_trace(d0, g) for g in grade_order]

fig = go.Figure(
    data=initial_traces,
    layout=go.Layout(
        title=dict(
            text="THE HARDEST BOULDER PROBLEM ASCENTS EVER",
            x=0.5,
            xanchor="center",
            font=dict(size=40, family="Arial Black"),
        ),
        template="plotly_dark",
        xaxis=dict(title="Date"),
        yaxis=dict(
            title="Grade",
            tickmode="array",
            tickvals=[17, 18, 19],
            ticktext=["8C+", "9A", "9A+"],
            range=[16.7, 19.3],
        ),
        legend=dict(title="Font Grade"),
        margin=dict(l=60, r=30, t=110, b=60),
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                x=0.0,
                y=-0.16,
                xanchor="left",
                yanchor="top",
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[
                            None,
                            {
                                "frame": {"duration": 450, "redraw": True},
                                "transition": {"duration": 0},
                                "fromcurrent": True,
                            },
                        ],
                    ),
                    dict(
                        label="Pause",
                        method="animate",
                        args=[
                            [None],
                            {
                                "frame": {"duration": 0, "redraw": False},
                                "mode": "immediate",
                                "transition": {"duration": 0},
                            },
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=len(months) - 1,
                x=0.08,
                y=-0.16,
                len=0.9,
                currentvalue=dict(prefix="Month: "),
                steps=[
                    dict(
                        method="animate",
                        label=month_label(m),
                        args=[
                            [month_label(m)],
                            {
                                "frame": {"duration": 0, "redraw": True},
                                "mode": "immediate",
                                "transition": {"duration": 0},
                            },
                        ],
                    )
                    for m in months
                ],
            )
        ],
    ),
    frames=[
        go.Frame(
            name=month_label(m),
            data=[make_trace(df[df["month"] <= m], g) for g in grade_order],
        )
        for m in months
    ],
)

# ---------- Pin the first ascent ----------
first = df.sort_values("date_parsed").iloc[0]
point_color = grade_colors.get(first["Grade_clean"], "#FFFFFF")

fig.add_annotation(
    x=first["date_parsed"],
    y=first["Y"],
    xref="x",
    yref="y",
    text=(
        f"<b>{first['Climber']}</b><br>"
        f"{first['Climb']}<br>"
        f"Grade: {first['Grade_clean']}<br>"
        f"Date: {first['Date']}"
    ),
    showarrow=True,
    arrowhead=2,
    ax=120,
    ay=-30,
    bgcolor=point_color,
    bordercolor="rgba(255,255,255,0.6)",
    borderwidth=1,
    borderpad=6,
    font=dict(color="black", size=12),
)

fig.update_xaxes(gridcolor="rgba(255,255,255,0.07)")
fig.update_yaxes(gridcolor="rgba(255,255,255,0.07)")

fig.show()

out_html = "hard_boulder_ascents_monthly.html"
fig.write_html(out_html)
print("Saved:", out_html)

Raw columns: ['#', 'Climber', 'Climb', 'Grade', 'Ascent Date']
Saved: hard_boulder_ascents_monthly.html


In [3]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# -------------------------------
# Data: first flash of each grade
# -------------------------------
dates = [
    datetime(2002, 9, 1),    # first V13 flash
    datetime(2011, 11, 28),  # first V14 flash
    datetime(2025, 7, 9),    # first V15 flash
]
grades = np.array([13, 14, 15], dtype=float)

def decimal_year(dt):
    year_start = datetime(dt.year, 1, 1)
    next_year = datetime(dt.year + 1, 1, 1)
    return dt.year + (dt - year_start).total_seconds() / (next_year - year_start).total_seconds()

years = np.array([decimal_year(d) for d in dates])

# -------------------------------
# Model 1: direct linear fit
# -------------------------------
coef_direct = np.polyfit(grades, years, 1)
poly_direct = np.poly1d(coef_direct)

# -------------------------------
# Model 2: linear growth in gaps
# -------------------------------
gaps = np.diff(years)
steps = np.array([1, 2], dtype=float)

coef_gap_linear = np.polyfit(steps, gaps, 1)
poly_gap_linear = np.poly1d(coef_gap_linear)

def predict_gap_linear(target_grade):
    predicted_years = [years[0]]
    for g in range(14, int(target_grade) + 1):
        step_idx = g - 13
        next_gap = poly_gap_linear(step_idx)
        predicted_years.append(predicted_years[-1] + next_gap)
    return np.array(predicted_years)

# -------------------------------
# Model 3: exponential growth
# -------------------------------
ratio = gaps[1] / gaps[0]

def predict_gap_exponential(target_grade):
    predicted_years = [years[0]]
    current_gap = gaps[0]

    for g in range(14, int(target_grade) + 1):
        step_idx = g - 13

        if step_idx == 1:
            next_gap = gaps[0]
        elif step_idx == 2:
            next_gap = gaps[1]
        else:
            current_gap = current_gap * ratio
            next_gap = current_gap

        predicted_years.append(predicted_years[-1] + next_gap)
        current_gap = next_gap

    return np.array(predicted_years)

# -------------------------------
# Build curves
# -------------------------------
x_all = np.array([13, 14, 15, 16], dtype=float)

y_direct = poly_direct(x_all)
y_gap_linear = predict_gap_linear(16)
y_gap_exp = predict_gap_exponential(16)

pred_direct = y_direct[-1]
pred_gap_linear = y_gap_linear[-1]
pred_gap_exp = y_gap_exp[-1]

# -------------------------------
# Plot
# -------------------------------
fig = go.Figure()

# actual data
fig.add_trace(go.Scatter(
    x=grades,
    y=years,
    mode="markers+text",
    name="Actual first flash",
    text=[
        "Sep 2002<br>V13",
        "Nov 28 2011<br>V14",
        "Jul 9 2025<br>V15"
    ],
    textposition=[
        "top center",     # V13
        "bottom right", # V14
        "top left"    # V15
    ],
    marker=dict(size=12),
))

# direct linear
fig.add_trace(go.Scatter(
    x=x_all,
    y=y_direct,
    mode="lines+markers",
    name=f"Direct linear fit (V16 ≈ {pred_direct:.1f})",
    line=dict(dash="dash"),
))

# linear gaps
fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_linear,
    mode="lines+markers",
    name=f"Linear-gap model (V16 ≈ {pred_gap_linear:.1f})",
    line=dict(dash="dot"),
))

# exponential gaps
fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_exp,
    mode="lines+markers",
    name=f"Exponential-gap model (V16 ≈ {pred_gap_exp:.1f})",
    line=dict(dash="longdash"),
))

# annotate V16 predictions
fig.add_annotation(x=16, y=pred_direct, text=f"{pred_direct:.1f}", showarrow=True, ax=40, ay=-20)
fig.add_annotation(x=16, y=pred_gap_linear, text=f"{pred_gap_linear:.1f}", showarrow=True, ax=40, ay=-40)
fig.add_annotation(x=16, y=pred_gap_exp, text=f"{pred_gap_exp:.1f}", showarrow=True, ax=40, ay=-60)

fig.update_layout(
    title="When Will the First V16 Boulder Be Flashed?",
    template="plotly_dark",
    xaxis=dict(
        title="Flash Grade",
        tickmode="array",
        tickvals=[13, 14, 15, 16],
        ticktext=["V13", "V14", "V15", "V16"],
        range=[12.7, 16.3],
    ),
    yaxis=dict(
        title="Year",
        range=[2000, max(pred_gap_exp, pred_gap_linear, pred_direct) + 2],
    ),
    legend=dict(
        title="Projection model",
        x=1.05,
        y=1,
        xanchor="left",
        yanchor="top",
    ),
    margin=dict(r=260),   # <- this is the important part
    width=1000,
    height=600,
)

fig.show()

In [31]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# -------------------------------
# Data: first consensus ascent of each hard grade
# -------------------------------
dates = [
    datetime(1992, 1, 1),   # V13 - La Danse des Balrogs
    datetime(1996, 1, 1),   # V14 - Radja
    datetime(2002, 8, 1),   # V15 - Monkey Wedding
    datetime(2010, 10, 1),  # V16 - Hypnotized Minds
    datetime(2016, 10, 1),  # V17 - Burden of Dreams
    datetime(2025, 11, 1),  # V18 - Exodia
]

grades = np.array([13,14,15,16,17,18], dtype=float)

labels = [
    "1992<br>V13",
    "1996<br>V14",
    "Aug 2002<br>V15",
    "Oct 2010<br>V16",
    "Oct 2016<br>V17",
    "Nov 2025<br>V18"
]

def decimal_year(dt):
    year_start = datetime(dt.year,1,1)
    next_year = datetime(dt.year+1,1,1)
    return dt.year + (dt-year_start).total_seconds()/(next_year-year_start).total_seconds()

years = np.array([decimal_year(d) for d in dates])

# -------------------------------
# Model 1: direct linear fit
# -------------------------------
coef_direct = np.polyfit(grades, years, 1)
poly_direct = np.poly1d(coef_direct)

# -------------------------------
# Model 2: linear growth in gaps
# -------------------------------
gaps = np.diff(years)
steps = np.arange(1,len(gaps)+1,dtype=float)

coef_gap_linear = np.polyfit(steps,gaps,1)
poly_gap_linear = np.poly1d(coef_gap_linear)

def predict_gap_linear(target_grade):
    predicted=[years[0]]
    for g in range(int(grades[0])+1,int(target_grade)+1):
        step=g-int(grades[0])
        gap=poly_gap_linear(step)
        predicted.append(predicted[-1]+gap)
    return np.array(predicted)

# -------------------------------
# Model 3: exponential gaps
# -------------------------------
log_gaps=np.log(gaps)
coef_gap_exp=np.polyfit(steps,log_gaps,1)

def predict_gap_exponential(target_grade):
    predicted=[years[0]]
    for g in range(int(grades[0])+1,int(target_grade)+1):
        step=g-int(grades[0])
        gap=np.exp(coef_gap_exp[1]+coef_gap_exp[0]*step)
        predicted.append(predicted[-1]+gap)
    return np.array(predicted)

# -------------------------------
# Build curves through V19
# -------------------------------
x_all=np.array([13,14,15,16,17,18,19],dtype=float)

y_direct=poly_direct(x_all)
y_gap_linear=predict_gap_linear(19)
y_gap_exp=predict_gap_exponential(19)

pred_direct=y_direct[-1]
pred_gap_linear=y_gap_linear[-1]
pred_gap_exp=y_gap_exp[-1]

# -------------------------------
# Plot
# -------------------------------
fig=go.Figure()

fig.add_trace(go.Scatter(
    x=grades,
    y=years,
    mode="markers+text",
    name="Actual first ascent",
    text=labels,
    textposition=[
    "top center",
    "top left",
    "bottom right",
    "top left",
    "bottom right",
    "top left"
],
    marker=dict(size=12)
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_direct,
    mode="lines+markers",
    name=f"Direct linear fit (V19 ≈ {pred_direct:.1f})",
    line=dict(dash="dash")
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_linear,
    mode="lines+markers",
    name=f"Linear-gap model (V19 ≈ {pred_gap_linear:.1f})",
    line=dict(dash="dot")
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_exp,
    mode="lines+markers",
    name=f"Exponential-gap model (V19 ≈ {pred_gap_exp:.1f})",
    line=dict(dash="longdash")
))

fig.add_annotation(x=19,y=pred_direct,text=f"{pred_direct:.1f}",showarrow=True,ax=40,ay=-20)
fig.add_annotation(x=19,y=pred_gap_linear,text=f"{pred_gap_linear:.1f}",showarrow=True,ax=60,ay=-40)
fig.add_annotation(x=19,y=pred_gap_exp,text=f"{pred_gap_exp:.1f}",showarrow=True,ax=40,ay=-60)

fig.update_layout(
    title="When Will the First V19 Boulder Be Climbed?",
    template="plotly_dark",
    xaxis=dict(
        title="Boulder Grade",
        tickmode="array",
        tickvals=[13,14,15,16,17,18,19],
        ticktext=["V13","V14","V15","V16","V17","V18","V19"],
        range=[12.7,19.3]
    ),
    yaxis=dict(
        title="Year",
        range=[1990,max(pred_direct,pred_gap_linear,pred_gap_exp)+2]
    ),
    legend=dict(
        title="Projection model",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top"
    ),
    margin=dict(r=260),
    width=1100,
    height=650
)

fig.show()

print(f"Direct linear fit V19: {pred_direct:.2f}")
print(f"Linear-gap model V19: {pred_gap_linear:.2f}")
print(f"Exponential-gap model V19: {pred_gap_exp:.2f}")

Direct linear fit V19: 2031.28
Linear-gap model V19: 2035.48
Exponential-gap model V19: 2035.71
